## Data Exploration

In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(r"c:\Users\USER\Python learning\teaching-code-material\week03_homework\data\car_price_sample.csv")
df.head()


,FuelType,CarPrice
0,Petrol,15200
1,Diesel,23500
2,Electric,41000
3,Hybrid,32000
4,Petrol,17800


In [16]:
# checking missing values
df[df['FuelType'].isnull()]

,FuelType,CarPrice
29,NaN,26500
38,NaN,33500


In [24]:
# average car price for each fuel type
df.groupby('FuelType')[['CarPrice']].mean().round(0)

,CarPrice
FuelType,
Diesel,23700.0
Electric,46278.0
Hybrid,32000.0
Petrol,18610.0


## Split the Data

In [28]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

In [29]:
# Define target and features
X = df[['FuelType']]
y = df['CarPrice']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print("\nTraining set shape:", X_train.shape)
print("\nTesting set shape:", X_test.shape)


Training set shape: (30, 1)

Testing set shape: (10, 1)


# Preparing the Data

In [36]:
#FuelType is categorical and it does not have a natural order
#One-hot encoding is the right choice because FuelType does not have a natural order

## Handling Missing Values in FuelType with `SimpleImputer` with the most frequent
imputer = SimpleImputer(strategy='most_frequent')
X_fuel = X_train[['FuelType']]
X_train_fuel_imputed = imputer.fit_transform(X_fuel)

X_train_fuel_imputed_df = pd.DataFrame(X_train_fuel_imputed, columns=['FuelType'], index=X_train.index)




In [37]:
# Handling Categorical Variables woth OneHOtEncoder
#categorical features = ['FuelType']


# Initializing the encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit encoder on imputed training FuelType and transform
X_train_fuel_encoded = encoder.fit_transform(X_train_fuel_imputed)

# Get the feature names for the new binary columns
encoded_columns = encoder.get_feature_names_out(['FuelType'])

# Converting back to a DataFrame for easier visualization
X_train_fuel_encoded_df = pd.DataFrame(X_train_fuel_encoded, columns=encoded_columns, index=X_train.index)

print("Original categorical feature (first 5 rows):")
print(X_train_fuel_imputed_df.head())
print("\nOne-hot encoded features (first 5 rows):")
print(X_train_fuel_encoded_df.head())


Original categorical feature (first 5 rows):
    FuelType
25    Diesel
9     Diesel
13    Diesel
31    Hybrid
34  Electric

One-hot encoded features (first 5 rows):
    FuelType_Diesel  FuelType_Electric  FuelType_Hybrid  FuelType_Petrol
25              1.0                0.0              0.0              0.0
9               1.0                0.0              0.0              0.0
13              1.0                0.0              0.0              0.0
31              0.0                0.0              1.0              0.0
34              0.0                1.0              0.0              0.0


## Training a Linear Regression model on training data

In [39]:
# Simple Linear Regression
model_simple = LinearRegression()

# Train the model on the one-hot encoded FuelType features and training target
model_simple.fit(X_train_fuel_encoded_df, y_train)

# Impute missing values in test FuelType
X_test_fuel_imputed = imputer.transform(X_test[['FuelType']])
X_test_fuel_imputed_df = pd.DataFrame(X_test_fuel_imputed, columns=['FuelType'], index=X_test.index)

# One-hot encode test FuelType
X_test_fuel_encoded = encoder.transform(X_test_fuel_imputed_df)
X_test_fuel_encoded_df = pd.DataFrame(X_test_fuel_encoded, columns=encoded_columns, index=X_test.index)

y_pred_s = model_simple.predict(X_test_fuel_encoded_df)

print(f"Simple LR Coefficients: {model_simple.coef_}")
print(f"Simple LR Intercept: {model_simple.intercept_}")
print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred_s):.2f}")

Simple LR Coefficients: [ -5428.0952381   16806.19047619    258.57142857 -11636.66666667]
Simple LR Intercept: 29908.095238095237
Mean Squared Error: 21982103.49


c:\Users\USER\Python learning\s26-redi-ml-Morisade\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(


In [41]:
# fuel type that has the highest coeficient
coefficients = pd.Series(model_simple.coef_, index=encoded_columns)
print("Coefficients for each FuelType category:")
print(coefficients)

# Identify fuel type with highest positive coefficient
highest_fueltype = coefficients.idxmax()
print(f"\nFuel type with highest positive coefficient: {highest_fueltype}")

Coefficients for each FuelType category:
FuelType_Diesel      -5428.095238
FuelType_Electric    16806.190476
FuelType_Hybrid        258.571429
FuelType_Petrol     -11636.666667
dtype: float64

Fuel type with highest positive coefficient: FuelType_Electric


# Answers to Concept questions

1) One-Hot Encoding was used because Fueltype column contains categorical variables and has no natural order.
- To my understanding this encoding converts each category into a separate binary column,
so that the linear regression model can process it as numerical data without assuming any orders. 

2) If we used the other encoding, it will give numerical order that does not exist and this leads to biased model.

3) Using .fit_transform() on test set may lead to fasle or incorrect prediction. Test set are only used for final evaluation.
